<a href="https://colab.research.google.com/github/CHAI24-ai/Modern-Automated-Self-Service-Fuel-Dispensing-Station/blob/main/Automated%20Fuel%20Vending%20Machine1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import time
from datetime import datetime

import numpy as np
from IPython.display import Audio, display
from gtts import gTTS

# -----------------------------
# Configuration / Constants
# -----------------------------

# Smaller capacity (100 L) so LOW alerts (≤ 20%) are easy to test
FUEL_TYPES = {
    "1": {"name": "Petrol", "price_per_liter": 105.0, "capacity_liters": 100.0},
    "2": {"name": "Diesel", "price_per_liter": 95.0, "capacity_liters": 100.0},
}

DISPENSE_RATE_LPS = 5            # Liters per second (simulation only)
MAX_SIMULATED_PUMP_TIME = 5      # seconds
ADMIN_PIN = "1234"               # Simple admin PIN for refill (for demo)
LOG_FILE = "fuel_log.txt"        # Transactions log file


class FuelVendingMachine:
    def __init__(self):
        # Copy initial capacities into a runtime dict (start full)
        self.available_fuel = {
            fuel_id: data["capacity_liters"] for fuel_id, data in FUEL_TYPES.items()
        }
        self.is_running = True

    # -----------------------------
    # Sound / alert helpers (Colab friendly)
    # -----------------------------
    def beep(self, times=1, delay=0.1, freq=800, dur=0.2):
        """
        Simple tone beep sound using sine wave (works in Colab).
        """
        sr = 22050  # sample rate (Hz)
        for _ in range(times):
            t = np.linspace(0, dur, int(sr * dur), False)
            tone = np.sin(2 * np.pi * freq * t)
            display(Audio(tone, rate=sr, autoplay=True))
            time.sleep(delay)

    def speak(self, text, lang="en"):
        """
        Voice alert using Google Text-to-Speech (gTTS).
        Saves a temporary mp3 and plays it.
        """
        try:
            tts = gTTS(text=text, lang=lang)
            filename = "voice_alert.mp3"
            tts.save(filename)
            display(Audio(filename, autoplay=True))
        except Exception as e:
            print(f"[VOICE ALERT ERROR] {e}")

    # -----------------------------
    # Basic UI helpers
    # -----------------------------
    def display_header(self):
        print("\n" + "=" * 50)
        print("   REMOTE AREA FUEL VENDING MACHINE SIMULATOR")
        print("=" * 50)

    def display_tank_levels(self):
        print("\n[ TANK LEVELS ]")
        for fuel_id, data in FUEL_TYPES.items():
            name = data["name"]
            current = self.available_fuel[fuel_id]
            capacity = data["capacity_liters"]

            # Percentage level
            percent = (current / capacity) * 100 if capacity > 0 else 0

            # Status message based on remaining level + sound + voice alert
            if current == 0:
                status = "EMPTY – refill immediately!"
                print(f"⚠ ALERT: {name} tank is COMPLETELY EMPTY! Refill required NOW!")

                # Beep + Voice alert
                self.beep(times=3, freq=600, dur=0.3)
                self.speak(f"Alert. {name} tank is empty. Refill immediately.")

            elif percent <= 20:
                status = "LOW – plan refill soon."
                print(f"⚠ WARNING: {name} tank is LOW! Only {percent:.1f}% remaining.")

                # Beep + Voice alert
                self.beep(times=2, freq=850, dur=0.35)
                self.speak(
                    f"Warning. {name} tank is low. Only {percent:.1f} percent fuel remaining. Please refill soon."
                )

            elif percent >= 95:
                status = "NEAR FULL / FULL."
            else:
                status = "OK."

            print(
                f" - {name}: {current:.1f} L / {capacity:.1f} L "
                f"({percent:5.1f}% ) -> {status}"
            )

    # -----------------------------
    # Safety & logging
    # -----------------------------
    def check_safety(self) -> bool:
        print("\n[SYSTEM CHECK] Running safety diagnostics...")
        time.sleep(0.5)
        print("[SYSTEM CHECK] All safety systems OK.\n")
        return True

    def log_transaction(self, fuel_name, liters, amount, balance_after):
        now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        log_entry = (
            f"{now} | Fuel: {fuel_name:7} | Liters: {liters:7.2f} | "
            f"Amount: ₹{amount:8.2f} | Remaining: {balance_after:8.2f} L\n"
        )
        try:
            with open(LOG_FILE, "a") as f:
                f.write(log_entry)
        except Exception as e:
            print(f"[LOG WARNING] Could not write to log file: {e}")

    # -----------------------------
    # Fuel selection & transaction
    # -----------------------------
    def select_fuel_type(self):
        print("\nSelect Fuel Type:")
        for fuel_id, data in FUEL_TYPES.items():
            print(
                f" {fuel_id}. {data['name']} "
                f"(Price: ₹{data['price_per_liter']:.2f} / L)"
            )

        choice = input("Enter option (1/2 or q to cancel): ").strip()
        if choice.lower() == "q":
            return None

        if choice not in FUEL_TYPES:
            print("Invalid fuel selection.")
            return None

        return choice

    def process_transaction(self):
        # 1. Safety check
        if not self.check_safety():
            print("ERROR: Safety fault detected. Cannot dispense fuel.")
            return

        # 2. Select fuel type
        fuel_choice = self.select_fuel_type()
        if fuel_choice is None:
            print("Transaction cancelled.\n")
            return

        fuel_info = FUEL_TYPES[fuel_choice]
        fuel_name = fuel_info["name"]
        price_per_liter = fuel_info["price_per_liter"]

        print(f"\nYou selected: {fuel_name}")
        print(f"Current price: ₹{price_per_liter:.2f} per liter")
        print(f"Available in tank: {self.available_fuel[fuel_choice]:.1f} L\n")

        # 3. Get amount from user
        amount_input = input("Enter amount to spend (e.g., 500.00) or 'q' to cancel: ₹")
        if amount_input.lower().strip() == "q":
            print("Transaction cancelled.\n")
            return

        try:
            amount_to_spend = float(amount_input)
            if amount_to_spend <= 0:
                print("Invalid amount. Must be greater than 0.")
                return
        except ValueError:
            print("Invalid input. Please enter a number.")
            return

        # 4. Calculate liters to dispense
        liters_to_dispense = amount_to_spend / price_per_liter

        # 5. Check available fuel
        if liters_to_dispense > self.available_fuel[fuel_choice]:
            max_liters = self.available_fuel[fuel_choice]
            max_amount = max_liters * price_per_liter
            print(
                f"\nInsufficient {fuel_name.lower()} in tank.\n"
                f"Maximum you can get now: {max_liters:.1f} L "
                f"(≈ ₹{max_amount:.2f})"
            )
            return

        # 6. Dispense fuel (simulation)
        print(
            f"\nDispensing {liters_to_dispense:.2f} L of {fuel_name} "
            f"for ₹{amount_to_spend:.2f}..."
        )
        self.dispense_fuel(liters_to_dispense)

        # 7. Update tank level
        self.available_fuel[fuel_choice] -= liters_to_dispense

        # 8. Print receipt-like summary
        print("\n--- TRANSACTION SUMMARY ---")
        print(f"Fuel Type     : {fuel_name}")
        print(f"Liters Given  : {liters_to_dispense:.2f} L")
        print(f"Amount Paid   : ₹{amount_to_spend:.2f}")
        print(f"Price / Liter : ₹{price_per_liter:.2f}")
        print(f"Remaining {fuel_name} in Tank: {self.available_fuel[fuel_choice]:.2f} L")
        print("---------------------------\n")
        print("Thank you for using the fuel vending machine!\n")

        # 9. Log to file
        self.log_transaction(
            fuel_name, liters_to_dispense, amount_to_spend, self.available_fuel[fuel_choice]
        )

    def dispense_fuel(self, liters):
        """
        Simulate the time taken to pump fuel.
        """
        time_to_pump = liters / DISPENSE_RATE_LPS if DISPENSE_RATE_LPS > 0 else 0
        time_to_sleep = min(time_to_pump, MAX_SIMULATED_PUMP_TIME)
        print("Pumping... please wait.")
        time.sleep(time_to_sleep)
        print("Pumping complete.")

    # -----------------------------
    # Admin / Maintenance
    # -----------------------------
    def admin_menu(self):
        pin = input("\nEnter admin PIN (or 'q' to cancel): ").strip()
        if pin.lower() == "q":
            print("Admin menu cancelled.")
            return

        if pin != ADMIN_PIN:
            print("Incorrect PIN. Access denied.")
            return

        while True:
            print("\n--- ADMIN MENU ---")
            print("1. View tank levels")
            print("2. Refill a tank")
            print("3. Back to main menu")
            choice = input("Enter option: ").strip()

            if choice == "1":
                self.display_tank_levels()

            elif choice == "2":
                self.refill_tank()

            elif choice == "3":
                print("Exiting admin menu.")
                break

            else:
                print("Invalid option. Try again.")

    def refill_tank(self):
        print("\nSelect fuel type to refill:")
        for fuel_id, data in FUEL_TYPES.items():
            print(f" {fuel_id}. {data['name']}")
        choice = input("Enter option: ").strip()

        if choice not in FUEL_TYPES:
            print("Invalid fuel type.")
            return

        fuel_info = FUEL_TYPES[choice]
        name = fuel_info["name"]
        capacity = fuel_info["capacity_liters"]
        current = self.available_fuel[choice]

        print(
            f"\nCurrent {name} level: {current:.1f} L "
            f"(Max capacity: {capacity:.1f} L)"
        )

        # Check if already full
        if current >= capacity:
            print(f"{name} tank is already FULL. You cannot add more fuel.")
            self.beep(times=2, freq=700, dur=0.2)
            self.speak(f"{name} tank is already full.")
            return

        amount_str = input("Enter liters to add: ").strip()
        try:
            to_add = float(amount_str)
            if to_add <= 0:
                print("Refill amount must be positive.")
                return
        except ValueError:
            print("Invalid number.")
            return

        # If trying to add more than available capacity, auto-adjust
        if current + to_add > capacity:
            allowed = capacity - current
            print(
                f"Cannot add {to_add:.1f} L. It exceeds max capacity.\n"
                f"Automatically adding only {allowed:.1f} L to reach full capacity."
            )
            to_add = allowed

        self.available_fuel[choice] += to_add
        print(
            f"{name} tank successfully refilled.\n"
            f"New level: {self.available_fuel[choice]:.1f} L"
        )

    # -----------------------------
    # Test Beep & Voice Menu
    # -----------------------------
    def test_beep_menu(self):
        print("\n--- TEST ALERT BEEP MENU ---")
        print("1. LOW fuel beep + voice")
        print("2. EMPTY tank beep + voice")
        print("3. FULL tank beep + voice")
        print("4. Back to main menu")
        choice = input("Enter option: ").strip()

        if choice == "1":
            print("Testing LOW fuel alert...")
            self.beep(times=2, freq=850, dur=0.35)
            self.speak("This is a test. Low fuel alert.")

        elif choice == "2":
            print("Testing EMPTY tank alert...")
            self.beep(times=3, freq=600, dur=0.3)
            self.speak("This is a test. Tank is empty.")

        elif choice == "3":
            print("Testing FULL tank alert...")
            self.beep(times=2, freq=700, dur=0.2)
            self.speak("This is a test. Tank is full.")

        elif choice == "4":
            return
        else:
            print("Invalid option.")

    # -----------------------------
    # Main loop
    # -----------------------------
    def run(self):
        while self.is_running:
            self.display_header()
            self.display_tank_levels()

            print("\nMain Menu:")
            print("1. New fuel transaction")
            print("2. Admin / Maintenance")
            print("3. Test beep & voice alerts")
            print("4. Exit machine")

            choice = input("Enter option: ").strip()

            if choice == "1":
                self.process_transaction()

            elif choice == "2":
                self.admin_menu()

            elif choice == "3":
                self.test_beep_menu()

            elif choice == "4":
                print("\nShutting down fuel vending machine simulator...")
                self.is_running = False

            else:
                print("Invalid option. Please try again.")


# -----------------------------
# Program entry point
# -----------------------------
if __name__ == "__main__":
    machine = FuelVendingMachine()
    machine.run()


   REMOTE AREA FUEL VENDING MACHINE SIMULATOR

[ TANK LEVELS ]
 - Petrol: 100.0 L / 100.0 L (100.0% ) -> NEAR FULL / FULL.
 - Diesel: 100.0 L / 100.0 L (100.0% ) -> NEAR FULL / FULL.

Main Menu:
1. New fuel transaction
2. Admin / Maintenance
3. Test beep & voice alerts
4. Exit machine
Enter option: 1

[SYSTEM CHECK] Running safety diagnostics...
[SYSTEM CHECK] All safety systems OK.


Select Fuel Type:
 1. Petrol (Price: ₹105.00 / L)
 2. Diesel (Price: ₹95.00 / L)
Enter option (1/2 or q to cancel): 1

You selected: Petrol
Current price: ₹105.00 per liter
Available in tank: 100.0 L

Enter amount to spend (e.g., 500.00) or 'q' to cancel: ₹100

Dispensing 0.95 L of Petrol for ₹100.00...
Pumping... please wait.
Pumping complete.

--- TRANSACTION SUMMARY ---
Fuel Type     : Petrol
Liters Given  : 0.95 L
Amount Paid   : ₹100.00
Price / Liter : ₹105.00
Remaining Petrol in Tank: 99.05 L
---------------------------

Thank you for using the fuel vending machine!


   REMOTE AREA FUEL VENDING MA


   REMOTE AREA FUEL VENDING MACHINE SIMULATOR

[ TANK LEVELS ]
 - Petrol: 99.0 L / 100.0 L ( 99.0% ) -> NEAR FULL / FULL.
 - Diesel: 100.0 L / 100.0 L (100.0% ) -> NEAR FULL / FULL.

Main Menu:
1. New fuel transaction
2. Admin / Maintenance
3. Test beep & voice alerts
4. Exit machine
Enter option: 3

--- TEST ALERT BEEP MENU ---
1. LOW fuel beep + voice
2. EMPTY tank beep + voice
3. FULL tank beep + voice
4. Back to main menu
Enter option: 3
Testing FULL tank alert...



   REMOTE AREA FUEL VENDING MACHINE SIMULATOR

[ TANK LEVELS ]
 - Petrol: 99.0 L / 100.0 L ( 99.0% ) -> NEAR FULL / FULL.
 - Diesel: 100.0 L / 100.0 L (100.0% ) -> NEAR FULL / FULL.

Main Menu:
1. New fuel transaction
2. Admin / Maintenance
3. Test beep & voice alerts
4. Exit machine
Enter option: 3

--- TEST ALERT BEEP MENU ---
1. LOW fuel beep + voice
2. EMPTY tank beep + voice
3. FULL tank beep + voice
4. Back to main menu
Enter option: 2
Testing EMPTY tank alert...



   REMOTE AREA FUEL VENDING MACHINE SIMULATOR

[ TANK LEVELS ]
 - Petrol: 99.0 L / 100.0 L ( 99.0% ) -> NEAR FULL / FULL.
 - Diesel: 100.0 L / 100.0 L (100.0% ) -> NEAR FULL / FULL.

Main Menu:
1. New fuel transaction
2. Admin / Maintenance
3. Test beep & voice alerts
4. Exit machine
Enter option: 3

--- TEST ALERT BEEP MENU ---
1. LOW fuel beep + voice
2. EMPTY tank beep + voice
3. FULL tank beep + voice
4. Back to main menu
Enter option: 2
Testing EMPTY tank alert...



   REMOTE AREA FUEL VENDING MACHINE SIMULATOR

[ TANK LEVELS ]
 - Petrol: 99.0 L / 100.0 L ( 99.0% ) -> NEAR FULL / FULL.
 - Diesel: 100.0 L / 100.0 L (100.0% ) -> NEAR FULL / FULL.

Main Menu:
1. New fuel transaction
2. Admin / Maintenance
3. Test beep & voice alerts
4. Exit machine
Enter option: 1

[SYSTEM CHECK] Running safety diagnostics...
[SYSTEM CHECK] All safety systems OK.


Select Fuel Type:
 1. Petrol (Price: ₹105.00 / L)
 2. Diesel (Price: ₹95.00 / L)
Enter option (1/2 or q to cancel): 2

You selected: Diesel
Current price: ₹95.00 per liter
Available in tank: 100.0 L

Enter amount to spend (e.g., 500.00) or 'q' to cancel: ₹500

Dispensing 5.26 L of Diesel for ₹500.00...
Pumping... please wait.
Pumping complete.

--- TRANSACTION SUMMARY ---
Fuel Type     : Diesel
Liters Given  : 5.26 L
Amount Paid   : ₹500.00
Price / Liter : ₹95.00
Remaining Diesel in Tank: 94.74 L
---------------------------

Thank you for using the fuel vending machine!


   REMOTE AREA FUEL VENDING MACHI


   REMOTE AREA FUEL VENDING MACHINE SIMULATOR

[ TANK LEVELS ]
 - Petrol: 99.0 L / 100.0 L ( 99.0% ) -> NEAR FULL / FULL.
 - Diesel: 94.7 L / 100.0 L ( 94.7% ) -> OK.

Main Menu:
1. New fuel transaction
2. Admin / Maintenance
3. Test beep & voice alerts
4. Exit machine
